# BTC Sentiment Collection — 1-Year Test + 2-Year Final

One notebook for both historical collection periods using the **same 3-agent strategy**:

1. **Collector Agent** — primary Bitcoin-news dataset + Alpha Vantage NEWS_SENTIMENT + GDELT
2. **Validator Agent** — date range, BTC relevance, de-duplication, quality checks, FinBERT
3. **Manager Agent** — resolves sentiment conflicts and produces article/hourly/daily datasets

### Only collection change
Alpha Vantage is now fetched in **7-day windows**, with **automatic recursive splitting** whenever a response approaches the API's 1,000-row limit. This prevents silent month-tail truncation.

### Sections
- **Section A — 1 year:** `2025-09-04 → 2026-09-04`
- **Section B — 2 years:** `2024-09-04 → 2026-09-04`

Both sections use the same shared functions below to avoid duplicated notebook code.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install -q pandas pyarrow requests transformers torch


## Shared configuration

Store `ALPHA_VANTAGE_API_KEY` in **Colab → Secrets (🔑)**.

For the corrected historical rebuild, this notebook deliberately refuses to run Alpha Vantage silently without a key. `HF_TOKEN` is optional but recommended for Hugging Face downloads.


In [ ]:
from google.colab import userdata
from pathlib import Path
import os, json, subprocess
import pandas as pd

ROOT = Path("/content/drive/MyDrive/btc sentiment year data")
SCRIPT = ROOT / "btc_sentiment_agents.py"

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

ALPHA_KEY = get_secret("ALPHA_VANTAGE_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

if ALPHA_KEY:
    os.environ["ALPHA_VANTAGE_API_KEY"] = ALPHA_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Collector script:", SCRIPT)
print("Alpha Vantage configured:", bool(ALPHA_KEY))
print("HF token configured:", bool(HF_TOKEN))

assert SCRIPT.exists(), f"Missing collector script: {SCRIPT}"
assert ALPHA_KEY, (
    "ALPHA_VANTAGE_API_KEY is required for this corrected historical rebuild. "
    "Add it in Colab Secrets and rerun this cell."
)


## Shared runner and validation

The validation checks file creation, timestamps, duplicates, count/share consistency, article-to-hour consistency, and source coverage diagnostics.

Important: an hour with no accepted article is not automatically considered a collection failure. Coverage is assessed at the **source/query level**, while the final model later represents verified no-news hours with `has_news=0`.


In [ ]:
EXPECTED_FILES = [
    "01_raw_collected.parquet",
    "02_validation_audit.parquet",
    "03_final_btc_sentiment_articles.parquet",
    "04_btc_sentiment_hourly.parquet",
    "05_btc_sentiment_daily.parquet",
    "run_manifest.json",
]

def run_collection(start_date, end_date, output_name, use_gdelt=True):
    output_dir = ROOT / output_name
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", str(SCRIPT),
        "--start-date", start_date,
        "--end-date", end_date,
        "--output-dir", str(output_dir),
    ]
    if not use_gdelt:
        cmd.append("--skip-gdelt")

    print("\nRunning:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return output_dir


def validate_collection(output_dir, requested_start, requested_end):
    output_dir = Path(output_dir)
    print("\n=== File check ===")
    for name in EXPECTED_FILES:
        p = output_dir / name
        print(f"{name}: {p.exists()}")
        assert p.exists(), f"Missing output: {p}"

    raw = pd.read_parquet(output_dir / "01_raw_collected.parquet")
    audit = pd.read_parquet(output_dir / "02_validation_audit.parquet")
    articles = pd.read_parquet(output_dir / "03_final_btc_sentiment_articles.parquet")
    hourly = pd.read_parquet(output_dir / "04_btc_sentiment_hourly.parquet")
    daily = pd.read_parquet(output_dir / "05_btc_sentiment_daily.parquet")

    for df in (raw, audit, articles, hourly, daily):
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

    start = pd.Timestamp(requested_start, tz="UTC")
    end = pd.Timestamp(requested_end, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

    assert raw["timestamp"].dropna().between(start, end).all()
    assert articles["timestamp"].dropna().between(start, end).all()
    assert hourly["timestamp"].dropna().between(start, end).all()
    assert hourly["timestamp"].duplicated().sum() == 0

    # Basic hourly integrity.
    count_sum = (
        hourly["positive_count"]
        + hourly["negative_count"]
        + hourly["neutral_count"]
    )
    assert (count_sum == hourly["article_count"]).all()

    share_sum = (
        hourly["positive_share"]
        + hourly["negative_share"]
        + hourly["neutral_share"]
    )
    assert (share_sum.sub(1.0).abs() < 1e-9).all()

    # Rebuild accepted-article counts by hour and compare.
    a = articles.copy()
    a["hour"] = a["timestamp"].dt.floor("h")
    article_counts = a.groupby("hour").size().rename("recomputed_count")
    check = hourly.set_index("timestamp")[["article_count"]].join(article_counts)
    assert check["recomputed_count"].notna().all()
    assert (check["article_count"] == check["recomputed_count"]).all()

    # Duplicate checks in accepted rows.
    duplicate_urls = (
        articles.loc[articles["url"].fillna("").ne(""), "url"].duplicated().sum()
    )
    duplicate_title_ts = articles.duplicated(
        subset=["timestamp", "title"], keep=False
    ).sum()

    print("\n=== Dataset summary ===")
    print("Raw rows:", len(raw))
    print("Accepted articles:", len(articles))
    print("Hourly news rows:", len(hourly))
    print("Daily news rows:", len(daily))
    print("Accepted article coverage:", articles["timestamp"].min(), "->", articles["timestamp"].max())
    print("Hourly news coverage:", hourly["timestamp"].min(), "->", hourly["timestamp"].max())
    print("Duplicate accepted URLs:", int(duplicate_urls))
    print("Duplicate timestamp+title rows:", int(duplicate_title_ts))

    print("\n=== Source coverage ===")
    for source_name, group in raw.groupby("source_dataset", dropna=False):
        g = group.dropna(subset=["timestamp"]).sort_values("timestamp")
        if len(g):
            print(
                f"{source_name}: rows={len(g):,} | "
                f"{g['timestamp'].min()} -> {g['timestamp'].max()}"
            )

    alpha = raw[raw["source_dataset"].eq("alphavantage")].copy()
    if alpha.empty:
        raise AssertionError(
            "No Alpha Vantage rows were collected. "
            "Do not use this run as the corrected historical rebuild."
        )

    # Month diagnostic: this does NOT demand an article every day.
    # It reveals whether a source repeatedly stops unusually early each month.
    alpha["month"] = alpha["timestamp"].dt.to_period("M")
    coverage = alpha.groupby("month")["timestamp"].agg(["min", "max", "count"])
    display(coverage)

    print("\nValidation passed: structure + article/hour aggregation are internally consistent.")
    print("Review the Alpha Vantage month coverage table before accepting the run for retraining.")

    return {
        "raw": raw,
        "audit": audit,
        "articles": articles,
        "hourly": hourly,
        "daily": daily,
        "alpha_month_coverage": coverage,
    }


# Section A — Corrected 1-Year Collection

This is the shorter validation/rebuild run.

**Period:** `2025-09-04 → 2026-09-04`  
**Output:** `data_1year_corrected`

Run this section first. If the Alpha Vantage coverage table no longer shows systematic mid-month stopping, proceed to Section B.


In [ ]:
ONE_YEAR_START = "2025-09-04"
ONE_YEAR_END = "2026-09-04"

one_year_dir = run_collection(
    ONE_YEAR_START,
    ONE_YEAR_END,
    "data_1year_corrected",
    use_gdelt=True,
)
one_year = validate_collection(
    one_year_dir,
    ONE_YEAR_START,
    ONE_YEAR_END,
)


# Section B — Final Corrected 2-Year Collection

Run this after Section A looks healthy.

**Period:** `2024-09-04 → 2026-09-04`  
**Output:** `data_2year_corrected`

This is the canonical sentiment source to use for the next two-year retraining after the final source/market alignment audit.


In [ ]:
TWO_YEAR_START = "2024-09-04"
TWO_YEAR_END = "2026-09-04"

two_year_dir = run_collection(
    TWO_YEAR_START,
    TWO_YEAR_END,
    "data_2year_corrected",
    use_gdelt=True,
)
two_year = validate_collection(
    two_year_dir,
    TWO_YEAR_START,
    TWO_YEAR_END,
)


## Final note before retraining

Do **not** point the training configs at `data_2year_corrected/04_btc_sentiment_hourly.parquet` until the final audit confirms:

- source coverage is no longer systematically truncated;
- BTC market timestamps align causally with sentiment hours;
- missing sentiment hours are represented as verified `has_news=0`;
- `coverage_available=0` is used wherever source coverage is genuinely uncertain;
- no look-ahead leakage exists for the 1h, 6h, and 24h targets.
